# Building AI Agents with LangGraph and OpenAI

You've likely interacted with AI models through various interfaces or packages and thought, "What if I could create an AI agent that understands my data?" Today, we'll embark on building such an application using LangGraph, a framework designed for developing applications powered by language models.

In this session, we'll create an AI agent driven by GPT-4 that can perform tasks based on your data. We'll cover the following steps:
- Installing LangGraph and its associated packages
- Exploring the dataset we'll be using
- Building tool calls for the AI agent
- Managing and storing state with LangGraph
- Querying and interacting with the AI agent
- Developing a basic application using LangGraph to "converse with your data"

## Before you begin

To get started with this project, you'll need a developer account for OpenAI. Follow the steps in the [getting-started.ipynb](https://www.datacamp.com/datalab/w/1625115b-0e86-4c4c-8210-a6650943bea3/edit/getting-started.ipynb) notebook to create an API key and store it in Workspace.

For this project, we will assume that you have already set the `OPENAI_API_KEY` environment variable.

In [ ]:
%%time
import warnings
warnings.filterwarnings("ignore")

%python3 -m pip install --upgrade pip

## Step 0: Setup

To perform this analysis, we need to install the following packages:

- `openai`: For interacting with OpenAI's API.
- `langchain`: A framework for developing applications with generative AI.
- `langchain-openai`: LangChain extension module for OpenAI functionality.
- `langchain-community`: LangChain extension module for community-driven features.
- `langgraph`: A package to orchestrate LLM systems.
- `typing_extensions`: Provides backports of new type system features to older versions of Python.

In [ ]:
%%time
# We tested the code-along with the following package versions
!pip install openai==1.63.2 \
             langchain==0.3.19 \
			 langchain-openai==0.3.6 \
			 langchain-community==0.3.18 \
             langgraph==0.2.74 \
             typing_extensions==4.12.2 ipython-autotime
%load_ext autotime

## Step 1: Explore The Data

In this project, we will analyze Airbnb listing data from various cities. The data is stored in the `data` directory and contains detailed information about each listing, including host details, location, room type, pricing, and availability. Here's a link to the source of the data: [Inside Airbnb](https://insideairbnb.com/).

We will iterate through each DataFrame individually and print out key information about them. This approach will help us understand the structure and contents of each dataset separately. Our ultimate goal is to create a LangGraph AI Agent that will assist us in exploring the data more efficiently. However, we will begin by manually exploring the data to understand its structure and contents.

### Instructions
1. Import `pandas` and `os`.
2. List all `.csv` files using `os.listdir`.
3. For each CSV file in the data directory, print out its name followed by the head of its data and some newlines.

In [1]:
# Import pandas and os


# Directory containing the data files
data_dir = 'data'

# List all files


# For all files, print out the head of the data


## Step 2: Define Function Calls

In this step, we will transform the exploratory data analysis (EDA) code from the previous step into reusable functions. These functions will act as tools for the Language Model (LLM) to interact with and analyze the data more efficiently. By encapsulating the EDA logic into functions, we enable the AI Agent we will build later to utilize these functions for effective data exploration.

### Instructions

1. Use `StringIO`, `redirect_stdout`, and the built-in `exec` to implement `exec_python_code`. Pass `globals` as a second argument to `exec` so it has access to the global scope (this will help with imports, etc.). The function should execute arbitrary Python code but record all output in a buffer that is returned as a string.
2. Implement the `list_csv_datasets` function, which works similarly to what you did in Step 1, instruction 2. It should return a string.
3. Implement the `get_dataset_details` function, which works similarly to what you did in Step 1, instruction 3. It should return a string. Note: you might have noticed before that `pandas` will only print out a limited set of columns. We want to avoid this for the function call, as it will mean the LLM can't see those columns. You can set the `display.max_columns` option to `None` using `pd.set_option`.

In [3]:
# Imports
from langchain_core.tools import tool
from io import StringIO
from contextlib import redirect_stdout
from IPython.display import display

# Tool call to execute random Python code
@tool
def exec_python_code(code: str) -> str:
    """Execute Python code.
    In order to show a pandas DataFrame, use the `display` function that is already loaded.
    In order to show a plot, import and use the plotly library. Use `display` to show the plot.
    In your Python code, you can access all of the datasets in the "data" folder. For example, you can use `pd.read_csv("data/listings_austin.csv")` to access the Austin listings dataset.
    
    Args:
        code: The data analysis code to execute
    """
    pass

# List CSV datasets
@tool
def list_csv_datasets() -> str:
    """Lists all CSV datasets"""
    pass

# Get the details of a CSV file
@tool
def get_dataset_details(path: str) -> str:
    """Gets details about a specific dataset.
    
    Args:
        path: The path to the CSV dataset
    """
    pass

## Step 3: Bind The LLM With Tools

In this step, we focus on integrating the Language Model (LLM) with various tools to enhance its capabilities. We have already created the tool functions: `list_csv_datasets`, `get_dataset_details`, and `exec_python_code`. By binding the LLM with these tools using Langchain, we can extend its functionality beyond basic text generation. This integration allows the LLM to perform tasks such as data retrieval, computation, and interaction with APIs, thereby making it more versatile and powerful in practical applications. We'll see how we can connect those functions to LangChain and try it out by invoking the LLM with a system prompt and user message.

### Instructions
1. Import `ChatOpenAI` from `langchain_openai.chat_models`
2. Initialize the LLM using `ChatOpenAI`. You can use `gpt-4o`, for example. Assign it to `llm` and use `llm.bind_tools` to bind the previously created functions. Assign to `llm_with_tools`.
3. Invoke `llm_with_tools` with the system prompt, wrapped in `SystemMessage`, and a message wrapped in `HumanMessage`.

In [4]:
# Import ChatOpenAI


# Create the LLM model and assign to llm, then bind the tools and assign to llm_with_tools


# Import SystemMessage and HumanMessage


system_prompt = "You are a helpful data analysis assistant. You can interact with the user directly or use function calls to help the user solve their data analysis tasks."

# Invoke the LLM with tools using system prompt and human message


## Step 4: Build The Graph

In this step, we'll use the LLM defined earlier to build an agent. We'll utilize a package called [LangGraph](https://www.langchain.com/langgraph) (`langgraph`) for this purpose. LangGraph interoperates with LangChain and facilitates the creation of more complex LLM workflows.

Consider the following graph:

<p align="center">
  <img src="images/graph.png" alt="LangGraph graph">
</p>

The graph begins with a `__start__` node, marking the entry point of the LLM workflow. Following this, there's an `analytics_agent` node responsible for handling the user's analytics queries by sending them to the LLM. As demonstrated earlier, the LLM can respond with a tool call. If the LLM issues a tool call, the control flow transitions from `analytics_agent` to the `tools` node following the conditional edge. This node manages the interaction with the tool function calls and sends the response back to the `analytics_agent`. If the `analytics_agent` ceases to issue tool calls, the workflow transitions to the `__end__` state.

This workflow's ability to interact with its environment through function calls and adapt based on observations from those function calls makes it an agentic workflow.

### Instructions

In this exercise, you will build an agent using the LangGraph package to handle analytics queries. Follow the steps below to construct the graph and define the necessary nodes and edges.

1. We've already added the imports for you. Check them out to know which packages we'll be using.
2. Create a class `State` that inherits from `TypedDict`. This will be the state backing the graph. Add a key `messages` to the `State` class, which should be a `list`. We will need to define a "reducer" on this state. This will hint LangGraph how the current state needs to be updated at each step. You can use the following type annotation for `messages` to achieve this: `Annotated[list, add_messages]`.
3. Implement a function `analytics_agent` that takes a `state` of type `State` as an argument. The function should return a dictionary with a key `messages`. Use the `llm_with_tools` to invoke a list of messages, starting with a `SystemMessage` and followed by the messages from the `state`.
4. Instantiate a `ToolNode` with the available tools.
5. Create an instance of `StateGraph` with `State` as the argument. Add the `analytics_agent` and `tools` nodes to the graph using the `add_node` method.
6. Use `add_conditional_edges` to connect the `analytics_agent` node to the `tools` node based on a condition. Add an edge from `START` to `analytics_agent` and from `tools` back to `analytics_agent`.
7. Compile the graph using the `compile` method of the graph builder. Invoke the graph with an initial state containing a user message asking for a map of Istanbul highlighting Airbnb prices.

In [5]:
# Imports
from typing_extensions import TypedDict
from typing import Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

# Define the state


# Define the analytics_agent node
def analytics_agent(state: State):
    # Implement
    return {}

# Define the tools node


# Build the graph and add the nodes


# Add the edges


# Compile the graph from the graph_builder and invoke it


Let's visualize the graph to compare it with the description above!

## Instructions
1. Import `display` and `Image` from `IPython.display`.
2. Display the PNG `Image` generated by `graph.get_graph().draw_mermaid_png()`.

In [6]:
# Import display and Image


# Draw the PNG


## Step 5: Use Memory

In this step, we will explore how to use memory in our language model integration to enhance its capabilities further. By incorporating memory, the model can retain information across interactions, allowing for more contextually aware and coherent responses over time.

### Instructions
1. Import `MemorySaver` from `langgraph.checkpoint.memory`.
2. Initialize a `MemorySaver` instance and assign it to a variable named `memory`.
3. Compile the graph using `graph_builder.compile(checkpointer=memory)` to enable memory usage in the language model.

In [7]:
# Import MemorySaver from langgraph.checkpoint.memory


# Initialize a MemorySaver instance


# Compile the graph using the MemorySaver instance to enable memory usage


## Step 6: Talk To Your Data

In this step, we will create a loop that allows us to interact with our data using the language model. This loop will enable continuous interaction, where the model can process input, generate responses, and update its memory for improved context awareness over time.

### Instructions
1. We have already made some imports for you and added a function to display an `AIMessage` nicely: `display_ai_message`.
2. Implement `interact_with_graph`:
    1. Define `memory_config`, which is a dictionary that can be passed as the second argument of `graph.stream`. It should define a `"thread_id"` for the memory.
    2. Define a `user_input_message`, which needs to be a `HumanMessage`.
    3. Stream the input to the graph defined above. Use `display_ai_message` on all the messages you receive as a response.
3. Execute the cell to enter the talk-to-your-data loop. You can stop by saying 'Bye' or by stopping the cell execution.

In [8]:
# Imports
import yaml
from IPython.display import display, Markdown
from langchain_core.messages import AIMessage, HumanMessage

# Function to nicely display a message
def display_ai_message(message):
    if not isinstance(message, AIMessage):
        return
    for tool_call in message.tool_calls:
        display(Markdown(f"""
**⚙️ Tool call: {tool_call['name']}**

<details><summary>Details</summary>{yaml.dump(tool_call['args'])}</details>
"""))
    if not isinstance(message.content, str) or message.content == '':
        return
    display(Markdown(message.content))

# Function to interact with LLM Agent
def interact_with_graph(user_input: str):
    # Define memory_config
    
    
    # Define user input message
    
    
    # Stream the input to the graph
    

# The talk-to-your-date loop
while True:
    try:
        display(Markdown("## User"))
        user_input = input()
        if user_input.lower().startswith("bye"):
            break
        display(Markdown("## Assistant"))
        interact_with_graph(user_input)
    except KeyboardInterrupt:
       break